# Generative AI Custom Evaluation
This is an example notebook which showcases how a user can use AI Core custom evaluation to benchmark their large language models, evaluate orchestration configuration or prompts for their use case.
It uses publicly available emanual.csv. The workload computes industry standard metrics to check the reliability of the response generate by llm.
<br>**Note: For detailed instructions please refer to [Readme](./Readme.md)**

# SetUp (Step 1)


In [ ]:
! pip install -r ../requirements.txt

## Load your environment variables

Ensure that your environment variables are set in a `.env` file (see sample.env for an example). If there is a missing field the notebook will prompt you for a value.

In [351]:
# Loading the credentials from the env file
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from dotenv import load_dotenv
import os

load_dotenv(override=True)


# Fetching environment variables or prompting the user if missing
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL") or input("AICORE_BASE_URL is missing. Please enter it: ")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP") or input("AICORE_RESOURCE_GROUP is missing. Please enter it (default: 'default'): ") or "default"
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL") or input("AICORE_AUTH_URL is missing. Please enter it: ")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID") or input("AICORE_CLIENT_ID is missing. Please enter it: ")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET") or input("AICORE_CLIENT_SECRET is missing. Please enter it: ")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY") or input("AWS_ACCESS_KEY is missing. Please enter it: ")
AWS_BUCKET_ID = os.getenv("AWS_BUCKET_ID") or input("AWS_BUCKET_ID is missing. Please enter it: ")
AWS_REGION = os.getenv("AWS_REGION") or input("AWS_REGION is missing. Please enter it: ")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY") or input("AWS_SECRET_ACCESS_KEY is missing. Please enter it: ")
DEPLOYMENT_URL = os.getenv("DEPLOYMENT_URL", None)
AWS_USERNAME = os.getenv("AWS_USERNAME")
AWS_HOST = os.getenv("AWS_HOST")

# Initializing the GenAIHubProxyClient
client = GenAIHubProxyClient(
    base_url=AICORE_BASE_URL,
    auth_url=AICORE_AUTH_URL,
    client_id=AICORE_CLIENT_ID,
    client_secret=AICORE_CLIENT_SECRET,
    resource_group=AICORE_RESOURCE_GROUP
)

# Dependencies and Helper Functions (Step 2)

In [ ]:
import os
import json



def get_dataset_file_name(folder_path):
    """
    Retrieves the name of the first file in the specified folder.
    """
    if not os.path.isdir(folder_path):
        print(f"The folder path '{folder_path}' does not exist.")
        return None

    items_in_folder = os.listdir(folder_path)

    for item in items_in_folder:
        item_path = os.path.join(folder_path, item)
        if os.path.isfile(item_path):
            return item

    print(f"No files were found in the folder '{folder_path}'.")
    return None



# --- MAIN EXECUTION ---
DATASET_FOLDER = "../DATASET"

DATASET_NAME = get_dataset_file_name(DATASET_FOLDER)

if DATASET_NAME:
    print(f"Dataset name: {DATASET_NAME}")
else:
    print("Missing run or dataset file.")
    raise SystemExit("Exiting due to missing run/dataset file.")


### Create a Bearer token

In [360]:
import requests
def create_token():
  
    payload = {
        'grant_type': 'client_credentials',
        'client_id': AICORE_CLIENT_ID,
        'client_secret':AICORE_CLIENT_SECRET
    }
    response = requests.post(AICORE_AUTH_URL, data=payload)
    print(f"Response status code: {response}")
    response_data = response.json()
    if 'access_token' in response_data:
        return response_data['access_token']
    else:
        raise Exception(f"Failed to get token: {response_data}")
token = create_token()
    

Response status code: <Response [200]>


### Create a Resource Group


If you already have a resource group provisioned with grounding and rag enabled, you can add the name fo your resource group at `user_resource_group_id`

**Note: the "labels" config is required to enable your resource group to use grounding and Rag metrics. Ensure you set the value to true**

In [ ]:
import requests
def create_resource_group():
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }
    resource_group_id =f"rag-notebook-test"
    api_url = f"{AICORE_BASE_URL}/v2/admin/resourceGroups"
    payload = {
        "resourceGroupId": resource_group_id,
        "labels": [
            {
            "key": "ext.ai.sap.com/document-grounding",
            "value": "true"
            }
        ]
    }
    response = requests.post(api_url, json=payload, headers=headers)
    if response.status_code == 202:
        return resource_group_id
    else:
        raise Exception(f"Failed to create resource group: {response.json()}")

user_resource_group_id = "" # add your provisioned resource group id here, if you have one
resource_group_id = user_resource_group_id or create_resource_group()
print(f"Resource Group created with ID: {resource_group_id}")

### Register an Object Store Secret
To use the evaluations service, you must register an object store with the name default. Optionally, you can register an additional object store with a name of your choice.

In [361]:
# setup authentication and headers needed for AI Core requests
def _get_headers():
    headers = {
        "Authorization": client.get_ai_core_token(),
        "AI-Resource-Group": resource_group_id,
        "Content-Type": "application/json",
    }
    return headers

In [ ]:
# Register S3 secret with AI Core which will be used an input source 
import requests
import json
import logging

def delete_oss_secret(oss_name=""):
    headers = _get_headers()
    
    DELETE_SECRETS_ENDPOINT = f'/v2/admin/objectStoreSecrets/{oss_name}'
    request_url = f"{AICORE_BASE_URL}{DELETE_SECRETS_ENDPOINT}"
    
    try:
        response = requests.delete(request_url, headers=headers, timeout=120)
        if response.status_code == 202:
            print(f"Successfully deleted object store secret: {oss_name}")
        elif response.status_code == 404:
            print(f"Object store secret not found: {oss_name}. It may not exist.")
        else:
            logging.error(f"Failed to delete object store secret: {oss_name}, Status Code: {response.status_code}")
    except Exception as e:
        logging.error(f"Error occurred while attempting to delete object store secret: {e}")
        raise

def register_oss_secret(oss_name="", path_prefix=""):
    headers = _get_headers()
    
    POST_SECRETS_ENDPOINT = '/v2/admin/objectStoreSecrets'
    request_url = f"{AICORE_BASE_URL}{POST_SECRETS_ENDPOINT}"
    
    request_body = {
        "name": oss_name,
        "data": {
            "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY,
            "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY
        },
        "type": "S3",
        "bucket": AWS_BUCKET_ID,
        "endpoint": "s3-eu-central-1.amazonaws.com",
        "region": AWS_REGION,
        "pathPrefix": path_prefix,
        "verifyssl": "0",
        "usehttps": "1",
    }
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        result = response.json()
        return result
    except:
        logging.error("Error occurred while attempting to create object store secret")
        raise
        
delete_oss_secret(oss_name="default")
delete_oss_secret(oss_name="genai-simplified-notebook")
        
register_oss_secret(oss_name="default", path_prefix="")
register_oss_secret(oss_name="genai-simplified-notebook", path_prefix="")

### Create a Grounding Secret

In the next step, we create a secret that enables grounding by adding on the "labels" config. This generic secret needs to be created to provide details of the hyperscaler and bucket details so that grounding service will know how to retrieve data from it.

In [ ]:
import time
import base64
def encode_base64(value):
    return base64.b64encode(value.encode('utf-8')).decode('utf-8')
  
def create_generic_secret():
    payload ={
    "name": "groundingsecret",
    "data": {
      "url": encode_base64("https://s3-eu-central-1.amazonaws.com"),                                  
      "authentication": encode_base64("NoAuthentication"),
      "description": encode_base64("grounding secret"),
      "access_key_id": encode_base64(AWS_ACCESS_KEY),
      "bucket": encode_base64(AWS_BUCKET_ID),
      "host": encode_base64(AWS_HOST),
      "region": encode_base64("eu-central-1"),
      "secret_access_key": encode_base64(AWS_SECRET_ACCESS_KEY),
      "username": encode_base64(AWS_USERNAME),
    },
    "labels": [
      {
        "key": "ext.ai.sap.com/document-grounding",
        "value": "true"
      },
      {
        "key": "ext.ai.sap.com/documentRepositoryType",
        "value": "S3"
      }
    ]
}
    time.sleep(60)
    try:
      headers = _get_headers()
      api_url = f"{AICORE_BASE_URL}/v2/admin/secrets"
      response = requests.post(api_url, headers=headers, json=payload)
      if(response.status_code == 200):
          print("Generic secret created successfully")
      else:
          print(f"Failed to create generic secret: {response}")
    except Exception as e:
          print(f"Error creating secret: {e}")
create_generic_secret()
    

### Create a Grounding Pipeline

This step creates the connection to where you have stored your grounding documents and allows a path to retrieve the documents during evaluation  

In [ ]:
def create_s3_grounding_pipeline():
    headers = _get_headers()
    api_url = f"{AICORE_BASE_URL}/v2/lm/document-grounding/pipelines"
    payload = {
        "type": "S3",
        "configuration": {
            "destination": "groundingsecret"
        }
    }
    time.sleep(5)  # Optional wait for secret availability

    try:
        response = requests.post(api_url, headers=headers, json=payload)
        if response.status_code == 201:
            print("S3 document grounding pipeline created successfully")
        else:
            print(f"Failed to create pipeline. Status: {response.status_code}, Response: {response.text}")
    except Exception as e:
        print(f"Error creating S3 document grounding pipeline: {e}")
create_s3_grounding_pipeline()

**Note: Check that the next step successfully runs to ensure you have set up properly**

In [ ]:
def test_get_retrieval_repository(headers):
    # To ensure chunking happens.
    api_url = f"{AICORE_BASE_URL}/v2/lm/document-grounding/retrieval/dataRepositories"

    try:
        response = requests.get(api_url, headers=headers)
        print("Check to see if the s3 is added in the body:", response.json())
        if response.status_code == 200:
            print("S3 document retrieval successfull")
        else:
            raise Exception(f"Failed to create pipeline. Status: {response.status_code}, Response: {response.text}")
    except Exception as e:
        raise Exception(f"Error creating S3 document grounding pipeline: {e}")
test_get_retrieval_repository(_get_headers())

In [ ]:
# uploading these files to Object store to register as an artifact inside ai core

import boto3
import os
import uuid

def upload_folder_to_s3(folder_path, bucket_name, s3_prefix=""):
    """
    Upload a folder to an S3 bucket recursively.

    :param folder_path: The local folder path to upload.
    :param bucket_name: The name of the S3 bucket.
    :param s3_prefix: Optional prefix to use for the S3 keys (e.g., subfolder in the bucket).
    """
    s3_client = boto3.client(
            's3',
            aws_access_key_id=AWS_ACCESS_KEY,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
            region_name=AWS_REGION
            )

    for root, dirs, files in os.walk(folder_path):
        for file_name in files:
            print("val of root is ", file_name)
            local_path = os.path.join(root, file_name)
            # Compute the relative path for the S3 key
            relative_path = os.path.relpath(local_path, folder_path)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")  # Ensure S3-compatible paths
            print("val of s3 key is ", s3_key)
            print(f"Uploading {local_path} to s3://{bucket_name}/{s3_key}")
            
            # Upload the file
            s3_client.upload_file(local_path, bucket_name, s3_key)

# Example usage
folder_to_upload_testdata = "../DATASET_RAG"
user_directory_prefix = "" # replace with your i-number as string here
prefix_guid = user_directory_prefix if user_directory_prefix is not None else str(uuid.uuid4().hex)
s3_testdata_prefix = f"genaiEvaluation/{prefix_guid}/testdata" # Leave empty for root of the bucket


upload_folder_to_s3(folder_to_upload_testdata, AWS_BUCKET_ID, s3_testdata_prefix)
input_artifact_path = f"ai://genai-simplified-notebook/genaiEvaluation/{prefix_guid}"

The user stores the input files in the object store and registers the root folder as artifact with AI Core. The File Upload and Artifact endpoints of AI Core API may be used for this purpose. In this example `genaiEvaluation\{prefix_guid}` is the root folder containing the orchestration configurations and test data which is registered as AI Core artifact.

In [ ]:
import requests
import logging
# Registering the uploaded files from AWS as artifacts to use inside configuration.

def register_artifact():
    headers = _get_headers()
    
    GET_ARTIFACTS_ENDPOINT = '/v2/lm/artifacts'
    request_url = f"{AICORE_BASE_URL}{GET_ARTIFACTS_ENDPOINT}"
    
    request_body = {
        "labels": [
            {
            "key": "ext.ai.sap.com/prompt-evaluation",
            "value": "true"
            }
        ],
        "name": "genai-eval-simplified-test-data",
        "kind": "other",
        "url": input_artifact_path, # input artifact path
        "description": "demo artifacts for evaluation flow.",
        "scenarioId": "genai-evaluations"
    }
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        result = response.json()
        print(result)
        return result['id']
    except:
        print("Error occurred while attempting to create an execution")
        raise
        

artifact_id = register_artifact()

## Create Orchestration Deployment
An orchestration Deployment URL is required for us to run our evaluation. Once created we need to wait until the deployment is running and provides us a deployment url which will be add to our configuration file in the next step. You can skip this step if you already have a orchestration deployment running.

In [ ]:
import requests
import json
import time



def create_orchestration_configuration():
    headers = _get_headers()
    request_body = {
    "name": "orchestrationDeployment",
    "executableId": "orchestration",
    "scenarioId": "orchestration",
    "parameterBindings": [
        {
            "key": "modelFilterList",
            "value": "null"
        },
        {
            "key": "modelFilterListType",
            "value": "allow"
        }
    ],
    "inputArtifactBindings": []
    }
    
    GET_CONFIGURATIONS_ENDPOINT = '/v2/lm/configurations'
    request_url = f"{AICORE_BASE_URL}{GET_CONFIGURATIONS_ENDPOINT}"
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        print(response)
        if(response.status_code != 201):
            raise
        result = response.json()
        print(result)
        return result['id']
    except:
        logging.error("Error occurred while attempting to create a Configuration")
        raise
    
def execute_orchestration_deployment(configuration_id):
    headers = _get_headers()
    GET_DEPLOYMENTS_ENDPOINT = '/v2/lm/deployments'
    request_url = f"{AICORE_BASE_URL}{GET_DEPLOYMENTS_ENDPOINT}"
    
    request_body = {
        "configurationId": configuration_id
    }
    
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        print(response)
        if(response.status_code != 202):
            print("Deployment execution failed")
        result = response.json()
        print(result)
        return result['id']
    
    except:
        logging.error("Error occurred while attempting to create an execution")
        raise

def get_deployment_status(orchestration_deployment_id):
    headers = _get_headers()
    api_url = f"{AICORE_BASE_URL}/v2/lm/deployments/{orchestration_deployment_id}?$select=status"
    timeout = 400  
    initial_interval = 30  
    pending_interval = 10
    start = time.time()

    status = None
    current_interval = initial_interval

    while time.time() - start < timeout:
        response = requests.get(api_url, headers=headers)
        if response.status_code == 200:
            status = response.json().get('status')
            print(f"Deployment {orchestration_deployment_id} status: {status}")
            # Adjust polling interval based on status
            if status == 'RUNNING':
                return True
            elif status == 'UNKNOWN':
                current_interval = initial_interval
            elif status == 'PENDING':
                current_interval = pending_interval

        else:
            print(f"Failed to fetch deployment status. HTTP {response.status_code}")
            return False

        # Waiting according to status for API call
        time.sleep(current_interval)

def get_deployment_url(orchestration_deployment_id):
    headers = _get_headers()
    response = requests.get(f"{AICORE_BASE_URL}/v2/lm/deployments/{orchestration_deployment_id}", headers=headers)
    if response.status_code != 200:
        raise Exception(f"Failed to get deployment URL: {response.status_code} - {response.text}")
    return response.json().get('deploymentUrl')

# You can skip this step if you already have a orchestration deployment running
deployment_url = DEPLOYMENT_URL
if not deployment_url:
    configuration_id = create_orchestration_configuration()
    orchestration_deployment_id = execute_orchestration_deployment(configuration_id)
    is_running = get_deployment_status(orchestration_deployment_id) 
    if is_running:
        deployment_url = get_deployment_url(orchestration_deployment_id)
        print(f"Deployment URL: {deployment_url}")
    else:
        print("Deployment is not running or failed.")

In [ ]:
# Manually set the orchestration deployment url
# deployment_url=""

## Select your Models
 
Add the LLMs you wish to use in the string `selected_models_str`



In [368]:
# Manual selection of models
selected_models_str="gpt-4o:2024-05-13"
print("Selected models string:", selected_models_str)

Selected models string: gpt-4o:2024-05-13


## Select system defined metrics
 
Add the system defined metrics you wish to use in the string `selected_metrics_str`.

**Note: If your dataset does not have a reference column, DO NOT Select metrics where reference is required.**

In [369]:
# Manual Selection of Metrics
selected_metrics_str = "Pointwise RAG Context Precision,Pointwise RAG Completeness"
print(selected_metrics_str)

Pointwise RAG Context Precision,Pointwise RAG Completeness


### Custom Metric Creation and Selection
This script checks for an evaluation metric in SAP AI Core.

1. You can provide Metric ID's directly by setting the variable as comma separated string:
       user_metric_ids = `"<your_metric_id>"`
   - ✅ If the ID exists, it will be returned.
  
2. You can create a new custom metric by adding the json in `custom_metric_list` string
   - The script will use the contents of the `custom_metric_list`
     to search for an existing metric by scenario + name + version.

3. If no existing metric is found:
   - A new metric will be created using the details in `custom_metric_list`.
   - Required fields in custom_metric: scenario, name, version, evaluationMethod.

4. At the end:
   - The script prints the final Metric ID that was found or created.

Note: Skip the two following cell if you do not want to create/select a custom metric for your workload

In [370]:
user_metric_ids = "d1868b00-1601-407a-92cd-0b9065682d1f,dbf56851-8444-45d3-a0c1-adbe210c7e771"

custom_metric_list = [
    {
    "name": "test-metric",
    "scenario": "genai-evaluations-test",
    "version": "0.0.1",
    "evaluationMethod": "llm-as-a-judge",
    "managedBy": "imperative",
    "systemPredefined": False,
    "metricType": "evaluation",
    "spec": {
      "outputType": "numerical",
      "promptType": "structured",
      "configuration": {
        "modelConfiguration": {
          "name": "gpt-4o",
          "version": "2024-05-13",
          "parameters": [
            {
              "key": "max_tokens",
              "value": "10000"
            }
          ]
        },
        "promptConfiguration": {
          "definition": "You will be assessing Groundedness (also known as Faithfulness), which measures whether the response relies solely on the provided context and avoids introducing external information or making claims not supported by it.",
          "evaluationTask": "You are an expert evaluator. Your task is to evaluate the groundedness of responses generated by AI models based on provided context.\nWe will provide you with the provided context (information the AI was supposed to use) and the AI-generated response. The original user query is also provided for background.\nYou should first read the provided context carefully, then evaluate if the response is fully supported by this context, based on the criteria provided in the Evaluation section below.\nYou will assign the response a rating following the Rating Rubric and Evaluation Steps.\nGive step-by-step explanations for your rating, and only choose ratings from the Rating Rubric.",
          "criteria": "Groundedness: Is all the information presented in the response verifiable against the provided context? Does the response avoid making claims or stating facts not present in the context?",
          "ratingRubric": [
            {
              "rating": 3,
              "rule": "Response is completely factual with no unsupported claims"
            },
            {
              "rating": 2,
              "rule": "Response has minor inaccuracies but no major contradictions"
            },
            {
              "rating": 1,
              "rule": "Response contains significant factual errors or hallucinations"
            }
          ]
        }
      }
    },
  "includeProperties": [
    "grounding_response"
  ],
  "additionalProperties": {
    "variables": [],
    "supported_values": [
      1,
      3
    ],
    "experimental": False
  }
}
]

In [ ]:
import os
import json
import requests


# --- Fetch all metrics from SAP AI Core ---
def fetch_all_metrics():
    request_url = f"{AICORE_BASE_URL}/v2/lm/evaluationMetrics"
    resp = requests.get(request_url, headers=_get_headers())
    resp.raise_for_status()
    return resp.json().get("resources", [])

# --- Create or fetch a metric ---
def create_or_get_metric(custom_metric, user_metric_id=None):
    all_metrics = fetch_all_metrics()

    # 1️⃣ User-supplied ID lookup
    if user_metric_id:
        for m in all_metrics:
            if m.get("id") == user_metric_id:
                print(f"✅ Metric already exists by ID: {user_metric_id}")
                return user_metric_id
        print(f"⚠️ User metric ID {user_metric_id} not found, will only include if valid later")

    # 2️⃣ Check by scenario, name, version
    scenario = custom_metric.get("scenario")
    name = custom_metric.get("name")
    version = custom_metric.get("version")
    if not all([scenario, name, version]):
        raise ValueError("Metric must include 'scenario', 'name', and 'version'")

    for m in all_metrics:
        if (m.get("scenario") == scenario and
            m.get("name") == name and
            m.get("version") == version):
            metric_id = m.get("id")
            print(f"✅ Metric already exists: {scenario}/{name} v{version}, ID = {metric_id}")
            return metric_id

    # 3️⃣ Create metric if not found
    request_url = f"{AICORE_BASE_URL}/v2/lm/evaluationMetrics"
    required_fields = ["scenario", "name", "version", "evaluationMethod", "metricType"]
    for f in required_fields:
        if f not in custom_metric:
            raise ValueError(f"❌ Missing required field: {f}")

    resp = requests.post(request_url, headers=_get_headers(), json=custom_metric)
    resp.raise_for_status()
    metric_id = resp.json().get("id")
    print(f"✅ Metric created successfully: {name} v{version}, ID = {metric_id}")
    return metric_id

# --- Main pipeline ---

# 1️⃣ Create/fetch metrics from SAP AI Core
metric_ids = []
for metric in custom_metric_list:
    try:
        print(f"metric:{metric}")
        metric_id = create_or_get_metric(metric)
        metric_ids.append(metric_id)
    except ValueError as e:
        print(f"Skipping metric due to error: {e}")

# 2️⃣ Validate user_metric_ids separately if provided
if user_metric_ids and user_metric_ids.strip():
    all_metrics = fetch_all_metrics()
    # Split comma-separated IDs and strip whitespace
    for uid in [uid.strip() for uid in user_metric_ids.split(",")]:
        if any(m.get("id") == uid for m in all_metrics):
            metric_ids.append(uid)
        else:
            print(f"⚠️ User metric ID {uid} does not exist in AI Core, skipping.")
# 3️⃣ Convert to comma-separated string
custom_metric_ids_str = ",".join(metric_ids)
print("✅ All processed metric IDs:", custom_metric_ids_str)

### Create Orchestration Registry Configuration

The following code defines a function `create_orchestration_registry_config()` that creates a new **Orchestration Configuration** in **Orchestration Registry**.

**Note** : If you wish to use an existing orchestration config, skip executing this cell and add the orchestration config id in `orchestration_registry_id` string in the next cell.

In [ ]:
def create_orchestration_registry_config():
    headers = _get_headers()
    
    CREATE_ORCHESTRATION_REGISTRY = '/v2/registry/v2/orchestrationConfigs'
    request_url = f"{AICORE_BASE_URL}{CREATE_ORCHESTRATION_REGISTRY}"
    model_name,model_version=selected_models_str.split(":")
    request_body = {
      "name": "genai-eval-test-1",
      "version": "0.0.1",
      "scenario": "genai-evaluations",
      "spec": {
        "modules": {
            "prompt_templating": {
                "prompt": {
                    "template": [
                        {
                        "role": "user",
                        "content": "You are a helpful assistant specialized in e-manual topics. Answer the following e-manual questions using the provided context. If the answer is not explicitly available in the context, respond with: `The answer is not available in the provided context.` \\n\\nRequest: {{?topic}}. \\n\\nContext: {{?groundingOutput}}"
                        }
                    ],
                    "defaults": {}
                },
                "model": {"name": f"{model_name}", "version": f"{model_version}",
            },
            },
            "grounding": {
                "type": "document_grounding_service",
                "config": {
                    "filters": [
                    {
                        "id": "helpRepo",
                        "data_repositories": [
                        "*"
                        ],
                        "search_config": {
                        "max_chunk_count": 10
                        },
                        "data_repository_type": "help.sap.com"
                    }
                    ],
                    "placeholders": {
                    "input": [
                        "topic"
                    ],
                    "output": "groundingOutput"
                    }
                }
            }
        }
      }
    }
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        if(response.status_code != 200):
            print(response.json())
            raise
        result = response.json()
        print(result)
        return result['id']
    except:
        logging.error("Error occurred while attempting to create a orchestration registry id")
        raise
orchestration_registry_id = create_orchestration_registry_config()

{'message': 'Orchestration config updated successfully.', 'id': '22ba2b67-ca81-41ab-989e-cd63a54a6499', 'scenario': 'genai-evaluations', 'name': 'genai-eval-test-1', 'version': '0.0.1'}


In [ ]:
# Manually set orchestration config id
# orchestration_registry_id=""

## Evaluation Configuration Creation

In [ ]:

import json
test_data_path = f"testdata/testdata/{DATASET_NAME}" # specify the test data path here. For the full folder just specifying testdata will work
test_datasets = json.dumps({'path': test_data_path, 'type': 'csv'})
print(test_datasets)
metrics_list = ",".join([selected_metrics_str,custom_metric_ids_str])
models_list = selected_models_str
print(f"Selected metrics: {metrics_list}")
print(f"Selected models: {models_list}")
#variable_mapping = json.dumps({'prompt/question': 'data/topic'}) # to map the question prompt variable to the entry in dataset.
# orchestration_deployment_url = deployment_url # needs to specify this to use a specific deployment id
orchestration_deployment_url = deployment_url
repetitions = "1"

In [ ]:
#  creating an AICORE Configuration.
import requests

request_body = {
    "name": "genai-eval-conf",
    "scenarioId": "genai-evaluations",
    "executableId": "genai-evaluations-simplified",
    "inputArtifactBindings": [
        {
            "key": "datasetFolder",
            "artifactId": artifact_id
        }
    ],
    "parameterBindings": [
        {
            "key": "repetitions",
            "value": repetitions
        },
        {
            "key": "orchestrationDeploymentURL",
            "value": orchestration_deployment_url
        },
        {
            "key": "metrics",
            "value": metrics_list
        },
        {
            "key": "testDataset",
            "value": test_datasets
        },
        {
            "key": "orchestrationRegistryIds",
            "value": orchestration_registry_id
        },
        {
            "key": "testRowCount",
            "value": "2"
        }
    ]
}

def create_aicore_configuration():
    headers = _get_headers()
    GET_CONFIGURATIONS_ENDPOINT = '/v2/lm/configurations'
    request_url = f"{AICORE_BASE_URL}{GET_CONFIGURATIONS_ENDPOINT}"
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        print(response)
        if(response.status_code != 201):
            raise
        result = response.json()
        print(result)
        return result['id']
    except:
        logging.error("Error occurred while attempting to create a Configuration")
        raise
        
configuration_id = create_aicore_configuration()

## Evaluation Execution Creation
Once Configration is create, we create the AI Core execution which triggers the evaluation workload.


In [ ]:
# create an execution with the created configuration.

import requests
def create_execution():
    headers = _get_headers()
    GET_EXECUTIONS_ENDPOINT = '/v2/lm/executions'
    request_url = f"{AICORE_BASE_URL}{GET_EXECUTIONS_ENDPOINT}"
    request_body = {"configurationId" : configuration_id} 
    try:
        response = requests.post(
            request_url, headers=headers, data=json.dumps(request_body), timeout=120
        )
        print("response received is ", response)
        result = response.json()
        print(result)
        return result['id']
    except:
        logging.error("Error occurred while attempting to create an execution")
        raise
 

execution_id = create_execution()

In [ ]:
# get execution status
import requests
def get_execution_status(execution_id):
    headers = _get_headers()
    LOG_EXECUTIONS_ENDPOINT = f'/v2/lm/executions/{execution_id}'
    request_url = f"{AICORE_BASE_URL}{LOG_EXECUTIONS_ENDPOINT}"
    try:
        response = requests.get(
            request_url, headers=headers, timeout=120
        )
        print("response received is ", response)
        result = response.json()
        return result
    except:
        logging.error("Error occurred while attempting to get execution status")
        raise
 

get_execution_status(execution_id)

<b>

1.  Run the following cells only when the status field in the Execution response is "COMPLETED" to view the results.
2.  The status field progresses through different states over time: UNKNOWN → PENDING → RUNNING → COMPLETED. Ensure it reaches COMPLETED before proceeding.


Note: The targetStatus will always be COMPLETED from the start, as it represents the intended final state of the Execution. Do not confuse it with the actual status field.
</b>

## Evaluation Result
The evaluation job produces two outputs
1. A SQLite DB file which stores the orchestration input, orchestration output, values for all the metrics calculated for this orchestration output and statistics such as latency for this orchestration output. These metric values are called raw metric values. This SQLite DB file is stored in the object store as an AI Core output artifact.
2. A set of metrics whose values are aggregated from the raw metric values. The aggregate metrics are stored in the tracking service. The user-defined tags along with the run names are stored with the metrics.
Post execution completion user can see the runs generated by the workload along with the aggregate metrics by calling the tracking api as show below

In [299]:
# Get aggregate metrics using execution id
import requests
def retrieve_aggregate_metrics(execution_id):
    headers = _get_headers()
    GET_METRICS_ENDPOINT = f'/v2/lm/metrics?tagFilters=evaluation.ai.sap.com/child-of={execution_id}'
    request_url = f"{AICORE_BASE_URL}{GET_METRICS_ENDPOINT}"
    try:
        response = requests.get(request_url, headers=headers, timeout=120)
        print("response received is ", response)
        result = response.json()
        return result
    except:
        logging.error("Error occurred while attempting to retreive aggeregate metrics for the run")
        raise

runs_data = retrieve_aggregate_metrics(execution_id)


response received is  <Response [200]>


To further drill down , User can also download the SQLite DB file from object storage and analyse the results(instance level metrics, logs etc) locally.

In [ ]:
# download the result artifacts from Object store.
import boto3

def download_all_objects(prefix, destination_folder):
    """
    Recursively download all objects from an S3 bucket starting with a specific prefix.

    :param bucket_name: Name of the S3 bucket.
    :param prefix: Prefix to filter objects in the bucket.
    :param destination_folder: Local folder to save the downloaded files.
    """
    s3_client = boto3.client(
            's3',
            aws_access_key_id=AWS_ACCESS_KEY,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
            region_name=AWS_REGION
            )

    # Ensure the destination folder exists
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    # Paginate through objects
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=AWS_BUCKET_ID, Prefix=prefix)

    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                local_file_path = os.path.join(destination_folder, os.path.relpath(key, prefix))

                # Ensure the local directory structure exists
                local_directory = os.path.dirname(local_file_path)
                if not os.path.exists(local_directory):
                    os.makedirs(local_directory)

                # Download the object
                print(f"Downloading {key} to {local_file_path}")
                s3_client.download_file(AWS_BUCKET_ID, key, local_file_path)


# Download the evaluation results from the object store. Look at execution status under "outputArtifacts" key to see the 'url'
# which shows the data path of where your output results are stored
EXECUTION_ID = execution_id
sqlite_db_prefix = f'{EXECUTION_ID}/tmp/'  # change the prefix based on where your output artifact is stored in the bucket.
destination_folder = 'results-new'

download_all_objects(sqlite_db_prefix, destination_folder)

<b>NOTE: The below Cell shows results of top 5 rows of the Evaluation Results across all SQLite tables. IF you wish to see all the entries you can comment the line saying df.head(5) in the below cell or modify the number accordingly.</b>

In [ ]:
# viewing the results from sqlite db in tabular format..
import sqlite3
import pandas as pd
from IPython.display import display, HTML

# Path to your SQLite database file
db_file = 'results-new/results.db'

connection = sqlite3.connect(db_file)

# Specify the table names you want to display
table_names = ['run','configuration', 'submission', 'submission_result', 'evaluation_result'] 

# Create the CSS and HTML container
html_content = """
<style>
/* Container to hold all tables */
.fixed-container {
    max-height: 600px;  /* Total container height */
    overflow-y: auto;   /* Vertical scroll for container */
    border: 2px solid #ddd;
    padding: 10px;
}

/* Table styling */
.table-container table {
    border-collapse: collapse; /* Merge table borders */
    width: 100%; /* Full width for tables */
    margin-bottom: 20px; /* Space between tables */
}

.table-container th, .table-container td {
    border: 1px solid #ddd; /* Add gridlines */
    text-align: left;       /* Align text to the left */
    padding: 8px;           /* Cell padding */
    white-space: nowrap;    /* Prevent text wrapping */
}

.table-container th {
    font-weight: bold;         /* Bold header text */
}

/* Allow dynamic column widths */
.table-container td {
    white-space: nowrap;   /* Prevent wrapping for long text */
}

</style>
<div class="fixed-container">
"""

for table_name in table_names:
    query = f"SELECT * FROM {table_name};"
    df = pd.read_sql_query(query, connection)
    # If you want to see all the rows across all tables, remove/comment the next line
    df = df.head(5)  # Limiting the number of rows displayed
    table_html = df.to_html(classes='table-container', index=False)
    html_content += f"""
    <div class="table-container">
        <h3>Table: {table_name}</h3>
        {table_html}
    </div>
    """

html_content += "</div>"

display(HTML(html_content))

# Close the connection
connection.close()

In [ ]:
#Delete Execution Id
def delete_execution():
    headers = _get_headers()
    EXEC_ID = execution_id
    GET_EXECUTIONS_ENDPOINT = '/v2/lm/executions/'
    request_url = f"{AICORE_BASE_URL}{GET_EXECUTIONS_ENDPOINT}{EXEC_ID}"
    try:
        response = requests.delete(
                request_url, headers=headers, params={"AI-Resource-Group":AICORE_RESOURCE_GROUP}, timeout=120
            )
        print(response)
        if(response.status_code != 202):
            raise
        result = response.json()
        print(result)
    except:
        logging.error("Error occurred while attempting to delete a Configuration")
        raise
    
delete_execution()